# DEM Conditioning: Training (Phase 2, Candidate 1)

Trains a genuinely new model -- same architecture family as `pcrtc/09`,
plus one new conditioning branch for DEM -- from scratch, on the exact
same spatial-block split as `09` (bit-identical shuffle, not just the
same patch set), so any metric difference is attributable specifically to
the DEM branch and nothing else. See `01_dem_acquisition_patch_extraction.ipynb`
for how the DEM patches were sourced (ArcticDEM, verified real coverage)
and this repo's `CONCEPTS.md`/`PHASE2_ARCHITECTURE_CANDIDATES.md` for why
this experiment is worth running now.

**This notebook does not execute automatically. Run cells top to bottom
on the GPU workstation, after `01` has been run and produced
`dem_patches_tuk/`.**

## GPU configuration

In [ ]:
import os
import sys
import json
import random
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler
import rasterio

assert torch.cuda.is_available(), 'CUDA is required. Run this notebook on the GPU environment.'
DEVICE = torch.device('cuda')
torch.backends.cudnn.benchmark = True
print('GPU:', torch.cuda.get_device_name(0))


## Paths and configuration

Every value here (`CONTEXT_K`, `SEED`, `VAL_FRACTION`, `BLOCK_SIZE_M`,
`BUFFER_M`, etc.) matches `pcrtc/09` exactly -- this is what makes the
split cell below reproduce `09`'s identical train/val assignment rather
than a merely-similar one. `DEM_FEAT_CH` is the one genuinely new
hyperparameter, controlling the new DEM encoder branch's output width.

In [ ]:
WORKING_REPO = Path('/cs/student/project_msc/2025/aibh/jiayiche')
TESSA_REPO = Path('/cs/student/project_msc/2025/aibh/jiayiche/tessa_baseline')
REGION = 'tuk'
LIDAR_DIR = WORKING_REPO / 'input_data' / 'lidar_patches_tuk_tessa'
S1_DIR = WORKING_REPO / 'input_data' / 's1_patches_tuk_pcrtc'
DEM_DIR = WORKING_REPO / 'input_data' / 'dem_patches_tuk'
CHECKPOINT_DIR = WORKING_REPO / 'checkpoints'
OUTPUT_DIR = WORKING_REPO / 's1_training_outputs'
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CHECKPOINT_NAME = f's1_{REGION}_pcrtc_dem_realattrs_spatialsplit_unet_best.pth'
METRICS_FILENAME = 's1_pcrtc_dem_realattrs_spatialsplit_validation_metrics.json'

CONTEXT_K = 3
TARGET_HW = (256, 256)
BATCH_SIZE = 8
EPOCHS = 100
TIMESTEPS = 1000
LEARNING_RATE = 1e-4
VAL_FRACTION = 0.15
SEED = 42
NOISE_SCHEDULE = 'linear'
ATTENTION_VARIANT = 'default'
LIDAR_SURVEY_DATE = __import__('datetime').date(2024, 4, 16)

# Spatial-block split params -- identical to 09, so the split itself is bit-identical
BLOCK_SIZE_M = 1024.0
BUFFER_M = 150.0

# New for this experiment only
DEM_FEAT_CH = 16


## Import Tessa's baseline implementation, and define the DEM-conditioned model

`AttrAwareSpatialPool` (inherited unchanged) is hard-coded to a
4-channel-per-view + per-view-attrs contract (`cond_img.view(B, k, 4, H, W)`)
-- confirmed by reading `src/model/unet.py` directly before writing this,
not assumed. A static DEM has neither temporal views nor per-view
attributes, so it cannot go through that pathway. `DEMConditionalUNet`
instead adds a small, independent encoder for the DEM raster, and
concatenates its output alongside the SAR-fused features right before
`input_conv` -- purely additive; the inherited SAR/attrs pathway
(`self.asap`, `self.downs`, `self.bottleneck_conv`, `self.ups`,
`self.output_conv`) is untouched.

In [ ]:
sys.path.insert(0, str(TESSA_REPO))
from src.model.unet import ConditionalUNet
from src.model.blocks import DoubleConv
from src.diffusion.utils import timestep_embedding
from src.diffusion.scheduler import LinearDiffusionScheduler, CosineDiffusionScheduler
from src.diffusion.sampling import p_sample_loop_ddim
from src.utils.recon_metrics import rmse, bias, sigma_error, normal_angle_error, average_jsd_multiscale, log_psd_rmse, zncc

def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(SEED)


class DEMConditionalUNet(ConditionalUNet):
    """ConditionalUNet + one additional, independent DEM conditioning branch."""

    def __init__(self, *args, dem_feat_ch=16, **kwargs):
        super().__init__(*args, **kwargs)
        self.dem_feat_ch = dem_feat_ch
        self.dem_encoder = nn.Sequential(
            nn.Conv2d(1, 16, 3, padding=1), nn.SiLU(),
            nn.Conv2d(16, dem_feat_ch, 3, padding=1), nn.SiLU(),
        )
        in_channels = kwargs.get('in_channels', 1)
        cond_channels = kwargs.get('cond_channels', 24)
        # Replace input_conv to account for the extra DEM feature channels
        self.input_conv = DoubleConv(in_channels + cond_channels + dem_feat_ch, self.base_channels, self.embed_dim)

    def forward(self, x, cond_img, attrs, dem, t):
        B = x.size(0)
        k = self.cond_k
        A_per = self.attr_dim_per

        t_emb = self.time_mlp(timestep_embedding(t, self.embed_dim))

        attrs_kA = attrs.view(B, k, A_per)
        fused_cond = self.asap(cond_img, attrs_kA, k=k)
        dem_feat = self.dem_encoder(dem)

        x = torch.cat([x, fused_cond, dem_feat], dim=1)

        skips = []
        x = self.input_conv(x, t_emb)
        for down in self.downs:
            skips.append(x)
            x = down(x, t_emb)

        x = self.bottleneck_conv(x, t_emb)

        for up in self.ups:
            skip = skips.pop()
            x = up(x, skip, t_emb)

        return self.output_conv(x)


class DemBoundModel(nn.Module):
    """Adapter: p_sample_loop_ddim calls model(x, cond, attrs, t) with a fixed
    4-arg signature, so the DEM has to be bound in rather than passed through.
    This lets the evaluation reuse Tessa's exact DDIM implementation unchanged
    (important -- a reimplemented sampler could differ subtly from the one 09
    was evaluated with, which would contaminate the comparison)."""

    def __init__(self, inner, dem):
        super().__init__()
        self.inner = inner
        self.dem = dem

    def forward(self, x, cond, attrs, t):
        return self.inner(x, cond, attrs, self.dem, t)


## Sentinel-1/LiDAR/DEM dataset adapter

Identical to `09`'s real-attrs dataset class, with one addition: also
loads the matching DEM patch (already collocated onto this exact LiDAR
patch's grid by `01`) and demeans it per-patch, same convention as the
LiDAR target -- absolute elevation carries no roughness information,
only the relative shape within the patch does.

In [ ]:
def build_real_attrs(s1_path, times, context_k):
    attrs_path = s1_path / 'attrs.json'
    attrs_list = json.load(open(attrs_path)) if attrs_path.exists() else []
    vecs = []
    for time_path in times:
        idx = int(time_path.stem[1:])
        a = attrs_list[idx] if idx < len(attrs_list) else {}
        if a.get('acquisition_date'):
            import datetime as dt
            acq_date = dt.date.fromisoformat(a['acquisition_date'])
            age_days = (acq_date - LIDAR_SURVEY_DATE).days
            age_norm = age_days / 30.0
        else:
            age_norm = 0.0
        orbit_dir = 1.0 if a.get('orbit_direction') == 'ASCENDING' else 0.0
        rel_orbit = (a.get('relative_orbit_number') or 0) / 175.0
        vecs.append([age_norm, orbit_dir, rel_orbit, 0.0, 0.0, 0.0, 0.0, 0.0])
    return torch.tensor(vecs, dtype=torch.float32).flatten()


class LidarS1DemDataset(Dataset):
    def __init__(self, s1_dir, lidar_dir, dem_dir, patch_ids, context_k=3, target_hw=(256, 256), train=False):
        self.s1_dir = Path(s1_dir)
        self.lidar_dir = Path(lidar_dir)
        self.dem_dir = Path(dem_dir)
        self.patch_ids = list(patch_ids)
        self.context_k = context_k
        self.target_hw = target_hw
        self.train = train

    def __len__(self):
        return len(self.patch_ids)

    def __getitem__(self, index):
        patch_id = self.patch_ids[index]
        lidar_path = self.lidar_dir / f'lidar_patch_{patch_id}.tif'
        s1_path = self.s1_dir / f's1_patch_{patch_id}'
        dem_path = self.dem_dir / f'dem_patch_{patch_id}.tif'

        with rasterio.open(lidar_path) as src:
            raw = src.read().astype(np.float32)
        target = raw[0]
        mask = (raw[1] > 0.5) if raw.shape[0] > 1 else np.isfinite(target)
        target = np.nan_to_num(target, nan=0.0, posinf=0.0, neginf=0.0)
        valid_count = max(1, int(mask.sum()))
        patch_mean = float(target[mask].sum() / valid_count)
        target = (target - patch_mean) * mask

        times = sorted(s1_path.glob('t*.tif'))[:self.context_k]
        if len(times) < self.context_k:
            raise RuntimeError(f'{s1_path} has fewer than {self.context_k} Sentinel-1 times')
        views = []
        for time_path in times:
            with rasterio.open(time_path) as src:
                sar = src.read()[:2].astype(np.float32)
            sar = np.nan_to_num(sar, nan=0.0, posinf=0.0, neginf=0.0)
            sar = np.maximum(sar, 1e-12)
            sar = 10.0 * np.log10(sar)
            sar_tensor = torch.from_numpy(sar).unsqueeze(0)
            sar_tensor = F.interpolate(sar_tensor, size=self.target_hw, mode='bilinear', align_corners=False).squeeze(0)
            sar_tensor = sar_tensor.repeat(2, 1, 1)
            views.append(sar_tensor)
        condition = torch.cat(views, dim=0)
        attrs = build_real_attrs(s1_path, times, self.context_k)

        with rasterio.open(dem_path) as src:
            dem_arr = src.read(1).astype(np.float32)
        dem_arr = np.nan_to_num(dem_arr, nan=0.0, posinf=0.0, neginf=0.0)
        dem_arr = dem_arr - dem_arr.mean()  # demean per-patch, same convention as the LiDAR target
        dem_tensor = torch.from_numpy(dem_arr).unsqueeze(0)
        if dem_tensor.shape[-2:] != torch.Size(self.target_hw):
            dem_tensor = F.interpolate(dem_tensor.unsqueeze(0), size=self.target_hw, mode='bilinear', align_corners=False).squeeze(0)

        return {
            'lidar': torch.from_numpy(target).unsqueeze(0).float(), 'mask': torch.from_numpy(mask),
            's1': condition.float(), 'attrs': attrs, 'dem': dem_tensor.float(),
            'patch_mean': torch.tensor(patch_mean), 'patch_id': patch_id,
        }


## Spatial-block split -- identical to `09`

Copied verbatim from `pcrtc/09`, not just "the same logic" -- same
`paired_ids` derivation (LiDAR ∩ S1, not filtered by DEM availability
first, since `01` already confirmed all 1676 have a DEM patch), same
seed, same block/shuffle order, so this produces the exact same train/val
patch assignment `09` trained on. Re-verifies zero leakage before
proceeding, same as `09` did.

In [ ]:
lidar_ids = {p.stem.split('_')[-1] for p in LIDAR_DIR.glob('lidar_patch_*.tif')}
s1_ids = {p.name.split('_')[-1] for p in S1_DIR.glob('s1_patch_*') if p.is_dir()}
paired_ids = sorted(lidar_ids & s1_ids)
assert paired_ids, 'No paired Sentinel-1/LiDAR patches found.'

dem_ids = {p.stem.split('_')[-1] for p in DEM_DIR.glob('dem_patch_*.tif')}
missing_dem = set(paired_ids) - dem_ids
assert not missing_dem, f'{len(missing_dem)} paired patches have no DEM patch -- re-run 01 before continuing.'

def patch_centroid(patch_id):
    with rasterio.open(LIDAR_DIR / f'lidar_patch_{patch_id}.tif') as src:
        b = src.bounds
    return ((b.left + b.right) / 2.0, (b.bottom + b.top) / 2.0)

centroids = {pid: patch_centroid(pid) for pid in paired_ids}

def block_id_and_boundary_distance(cx, cy, block_size):
    bx, by = int(cx // block_size), int(cy // block_size)
    dx = min(cx - bx * block_size, (bx + 1) * block_size - cx)
    dy = min(cy - by * block_size, (by + 1) * block_size - cy)
    return (bx, by), min(dx, dy)

blocks = {}
dropped_buffer = []
for pid, (cx, cy) in centroids.items():
    bid, boundary_dist = block_id_and_boundary_distance(cx, cy, BLOCK_SIZE_M)
    if boundary_dist < BUFFER_M:
        dropped_buffer.append(pid)
        continue
    blocks.setdefault(bid, []).append(pid)

block_ids = list(blocks.keys())
random.Random(SEED).shuffle(block_ids)

target_val_patches = int(len(paired_ids) * VAL_FRACTION)
val_ids, train_ids = [], []
running_val_count = 0
for bid in block_ids:
    if running_val_count < target_val_patches:
        val_ids.extend(blocks[bid])
        running_val_count += len(blocks[bid])
    else:
        train_ids.extend(blocks[bid])

print(f'Spatial-block split (matching pcrtc/09): train={len(train_ids)}, val={len(val_ids)} '
      f'(target val={target_val_patches}, {len(dropped_buffer)} dropped as buffer)')

from shapely.geometry import box
from shapely.strtree import STRtree

train_boxes = []
for pid in train_ids:
    with rasterio.open(LIDAR_DIR / f'lidar_patch_{pid}.tif') as src:
        train_boxes.append(box(*src.bounds))
val_boxes = []
for pid in val_ids:
    with rasterio.open(LIDAR_DIR / f'lidar_patch_{pid}.tif') as src:
        val_boxes.append((pid, box(*src.bounds)))
tree = STRtree(train_boxes)
overlap_count = 0
for pid, vbox in val_boxes:
    hits = tree.query(vbox)
    real_overlaps = [h for h in hits if train_boxes[h].intersects(vbox) and not train_boxes[h].touches(vbox)]
    if real_overlaps:
        overlap_count += 1
assert overlap_count == 0, 'Spatial-block split has leakage -- something has changed vs. 09, stop and investigate.'
print(f'Confirmed: {overlap_count} / {len(val_ids)} validation patches overlap a training patch. Leakage-free.')

train_dataset = LidarS1DemDataset(S1_DIR, LIDAR_DIR, DEM_DIR, train_ids, CONTEXT_K, TARGET_HW, train=True)
val_dataset = LidarS1DemDataset(S1_DIR, LIDAR_DIR, DEM_DIR, val_ids, CONTEXT_K, TARGET_HW, train=False)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)


## Model, scheduler, optimizer

In [ ]:
model = DEMConditionalUNet(
    in_channels=1, cond_channels=4 * CONTEXT_K, attr_dim=8 * CONTEXT_K, base_channels=128,
    embed_dim=256, unet_depth=4, attention_variant=ATTENTION_VARIANT, cond_k=CONTEXT_K,
    dem_feat_ch=DEM_FEAT_CH,
).to(DEVICE)
scheduler = LinearDiffusionScheduler(TIMESTEPS, device=DEVICE) if NOISE_SCHEDULE == 'linear' else CosineDiffusionScheduler(TIMESTEPS, device=DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
n_params = sum(p.numel() for p in model.parameters())
print(f'Model parameters: {n_params:,} (09 had ~104.6M; this should be slightly more, from the new DEM encoder + wider input_conv)')


## Training loop

In [ ]:
def masked_mse(prediction, target, mask):
    valid = mask.bool().unsqueeze(1)
    error = (prediction - target) ** 2
    return error[valid].mean()

scaler = GradScaler()
history = {'train_loss': [], 'val_loss': []}
best_val = float('inf')
for epoch in range(EPOCHS):
    model.train()
    train_total = 0.0
    for batch in train_loader:
        target = batch['lidar'].to(DEVICE, non_blocking=True)
        condition = batch['s1'].to(DEVICE, non_blocking=True)
        attrs = batch['attrs'].to(DEVICE, non_blocking=True)
        dem = batch['dem'].to(DEVICE, non_blocking=True)
        mask = batch['mask'].to(DEVICE, non_blocking=True)
        timestep = torch.randint(0, TIMESTEPS, (target.size(0),), device=DEVICE)
        optimizer.zero_grad(set_to_none=True)
        with autocast():
            noisy = scheduler.q_sample(target, timestep)
            prediction = model(noisy, condition, attrs, dem, timestep)
            loss = masked_mse(prediction, target, mask)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        train_total += loss.item()
    model.eval()
    val_total = 0.0
    with torch.no_grad():
        for batch in val_loader:
            target = batch['lidar'].to(DEVICE, non_blocking=True)
            condition = batch['s1'].to(DEVICE, non_blocking=True)
            attrs = batch['attrs'].to(DEVICE, non_blocking=True)
            dem = batch['dem'].to(DEVICE, non_blocking=True)
            mask = batch['mask'].to(DEVICE, non_blocking=True)
            timestep = torch.randint(0, TIMESTEPS, (target.size(0),), device=DEVICE)
            with autocast():
                prediction = model(scheduler.q_sample(target, timestep), condition, attrs, dem, timestep)
                val_total += masked_mse(prediction, target, mask).item()
    train_loss = train_total / max(1, len(train_loader))
    val_loss = val_total / max(1, len(val_loader))
    history['train_loss'].append(train_loss); history['val_loss'].append(val_loss)
    print(f'Epoch {epoch + 1:03d}/{EPOCHS}: train={train_loss:.6f} val={val_loss:.6f}')
    if val_loss < best_val:
        best_val = val_loss
        torch.save({'model_state_dict': model.state_dict(), 'config': {'context_k': CONTEXT_K, 'timesteps': TIMESTEPS, 'noise_schedule': NOISE_SCHEDULE, 'region': REGION, 'dem_feat_ch': DEM_FEAT_CH}, 'epoch': epoch + 1, 'val_loss': val_loss}, CHECKPOINT_DIR / CHECKPOINT_NAME)


## Evaluation on the validation split

In [ ]:
best_path = CHECKPOINT_DIR / CHECKPOINT_NAME
checkpoint = torch.load(best_path, map_location=DEVICE)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()
sampler = p_sample_loop_ddim
metric_rows = []
example_patches = []
N_EXAMPLES = 6
with torch.no_grad():
    for batch in val_loader:
        target = batch['lidar'].to(DEVICE)
        condition = batch['s1'].to(DEVICE)
        attrs = batch['attrs'].to(DEVICE)
        dem = batch['dem'].to(DEVICE)
        mask = batch['mask'].to(DEVICE).bool()
        bound = DemBoundModel(model, dem)
        prediction = sampler(bound, scheduler, target.shape, condition, attrs, DEVICE)
        means = batch['patch_mean'].to(DEVICE).view(-1, 1, 1, 1)
        gt_absolute = target + means
        pred_absolute = prediction + means
        for i, patch_id in enumerate(batch['patch_id']):
            gt_i, pred_i, mask_i = gt_absolute[i], pred_absolute[i], mask[i]
            gt_valid = gt_i.squeeze()[mask_i].cpu().numpy()
            pred_valid = pred_i.squeeze()[mask_i].cpu().numpy()
            row = {
                'patch_id': patch_id,
                'rmse_m': float(rmse(gt_i, pred_i, mask_i).item()),
                'bias_m': float(bias(gt_i, pred_i, mask_i).item()),
                'sigma_error_pct': float(sigma_error(gt_i, pred_i, mask_i).item()),
                'normal_angle_error_deg': float(normal_angle_error(gt_i, pred_i, mask_i, pixel_size=1.0, degrees=True).item()),
                'jsd': float(average_jsd_multiscale(gt_i, pred_i, pixel_size=1.0, mask=mask_i).item()),
                'psd_rmse': float(log_psd_rmse(gt_i, pred_i, pixel_size=1.0, mask=mask_i).item()),
                'zncc': float(zncc(gt_i, pred_i, mask_i).item()),
                'gt_std_val': float(gt_valid.std()) if gt_valid.size > 0 else float('nan'),
                'pred_std_val': float(pred_valid.std()) if pred_valid.size > 0 else float('nan'),
            }
            metric_rows.append(row)
            if len(example_patches) < N_EXAMPLES:
                example_patches.append({
                    'patch_id': patch_id, 'gt': gt_i.squeeze().cpu().numpy(), 'pred': pred_i.squeeze().cpu().numpy(),
                    'mask': mask_i.squeeze().cpu().numpy(),
                })
metrics_path = OUTPUT_DIR / METRICS_FILENAME
with metrics_path.open('w') as handle:
    json.dump(metric_rows, handle, indent=2)
print('Saved:', metrics_path)
print('Mean metrics:', {key: float(np.nanmean([row[key] for row in metric_rows])) for key in metric_rows[0] if key != 'patch_id'})


## Comparison against `09` (identical validation patches, DEM branch is the only difference)

In [ ]:
original_metrics_path = OUTPUT_DIR / 's1_pcrtc_realattrs_spatialsplit_validation_metrics.json'
if original_metrics_path.exists():
    orig_rows = json.load(open(original_metrics_path))
    orig_mean = {k: float(np.nanmean([r[k] for r in orig_rows])) for k in orig_rows[0] if k != 'patch_id'}
    new_mean = {key: float(np.nanmean([row[key] for row in metric_rows])) for key in metric_rows[0] if key != 'patch_id'}
    print(f'{"metric":<20}{"09 (no DEM)":>16}{"+ DEM":>16}')
    for k in new_mean:
        if k in orig_mean:
            print(f'{k:<20}{orig_mean[k]:>16.4f}{new_mean[k]:>16.4f}')
else:
    print('09\'s validation metrics file not found -- compare manually against the numbers already documented for 09.')
